# Here is the code to run with files stored in GCS

In [1]:
import argparse
import os

from pyspark.sql import SparkSession

In [ ]:
# Parse command-line arguments
parser = argparse.ArgumentParser()
parser.add_argument(
    "--is-local",
    type=str,
    default="true",
    help="Whether running in local mode",
)
args, unknown = parser.parse_known_args()
is_local = args.is_local.lower() == "true"

print(f"Is local environment: {is_local}")

# Build SparkSession with conditional configuration
if is_local:
    adc_path = os.path.expanduser(
        "~/.config/gcloud/application_default_credentials.json"
    )
    gcs_connector_jar_path = "./gcs-connector-hadoop3-latest.jar"
    if not os.path.exists(gcs_connector_jar_path):
        raise FileNotFoundError(
            f"GCS connector JAR not found at {gcs_connector_jar_path}"
        )

    sparkgcp = (
        SparkSession.builder.appName("LocalGCS")
        .config("spark.driver.host", "localhost")
        .config(
            "spark.jars",
            gcs_connector_jar_path,
        )
        .config(
            "spark.hadoop.google.cloud.auth.service.account.enable", "true"
        )
        .config(
            "spark.hadoop.google.cloud.auth.service.account.json.keyfile",
            adc_path,
        )
        .config(
            "spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        )
        .config(
            "spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS",
        )
        .getOrCreate()
    )
else:
    sparkgcp = SparkSession.builder.appName("GCSCluster").getOrCreate()

sc = sparkgcp.sparkContext
sc.setLogLevel("WARN")

print("✅ Connected to Spark cluster!")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")

gcs_path = "gs://msds-694-cohort-14-3/data/num_2020.csv"
gcs_rdd = sc.textFile(gcs_path)
print(f"GCS RDD count: {gcs_rdd.count()}")


Is local environment: True


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/30 22:52:27 WARN Utils: Your hostname, Ignacios-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.231 instead (on interface en0)
25/11/30 22:52:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/11/30 22:52:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/30 22:52:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/30 22:52:28 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/30 22:52:28 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/30 22:52:28 WA

✅ Connected to Spark cluster!
Spark Version: 4.0.1
Master: local[*]
App ID: local-1764571948565
